In [4]:
import functools
import time
import json
import random
import hashlib
from pathlib import Path

In [5]:
WORKDIR = Path("./_workspace")
WORKDIR.mkdir(exist_ok=True)

In [6]:
def retry(max_attempts: int = 3, delay: float = 0.1, backoff: float = 2.0):
    def opakuj(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            ostatni_blad = None

            for proba in range(max_attempts):
                try:
                    return func(*args, **kwargs)
                except Exception as blad:
                    ostatni_blad = blad

                    if proba < max_attempts - 1:
                        czas = delay * (backoff ** proba)
                        print(f"Nie udało się: {blad}. Próbuję jeszcze raz...")
                        time.sleep(czas)

            raise ostatni_blad

        return wrapper
    return opakuj

In [7]:
def cache_to_disk(cache_dir: Path):
    cache_dir.mkdir(exist_ok=True, parents=True)

    def opakuj(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            dane_klucza = {
                "function": func.__name__,
                "args": args,
                "kwargs": kwargs
            }

            tekst_klucza = repr(dane_klucza)
            klucz = hashlib.md5(tekst_klucza.encode("utf-8")).hexdigest()
            plik_cache = cache_dir / f"{klucz}.json"

            if plik_cache.exists():
                print("Wynik jest już w cache, więc nie uruchamiam funkcji ponownie.")
                with open(plik_cache, "r", encoding="utf-8") as f:
                    return json.load(f)

            print("Nie ma wyniku w cache, więc uruchamiam funkcję.")
            wynik = func(*args, **kwargs)

            with open(plik_cache, "w", encoding="utf-8") as f:
                json.dump(wynik, f, ensure_ascii=False, indent=2)

            return wynik

        return wrapper
    return opakuj

In [8]:
@cache_to_disk(WORKDIR / "flaky_cache")
@retry(max_attempts=5, delay=0.05, backoff=2.0)
def flaky_fetch(text_id: int) -> dict:
    if random.random() < 0.5:
        raise ValueError(f"udawany błąd sieci dla id={text_id}")
    return {"id": text_id, "text": f"przykład {text_id}"}


print("Pierwsze wywołanie dla id=1:")
print(flaky_fetch(1))

print("\nDrugie wywołanie dla id=1:")
print(flaky_fetch(1))

Pierwsze wywołanie dla id=1:
Wynik jest już w cache, więc nie uruchamiam funkcji ponownie.
{'id': 1, 'text': 'przyklad 1'}

Drugie wywołanie dla id=1:
Wynik jest już w cache, więc nie uruchamiam funkcji ponownie.
{'id': 1, 'text': 'przyklad 1'}


In [9]:
sukcesy = 0
porazki = 0

for i in range(100):
    try:
        flaky_fetch(i)
        sukcesy += 1
    except ValueError:
        porazki += 1

p_empiryczne = sukcesy / 100
p_teoretyczne = 1 - 0.5 ** 5

Wynik jest już w cache, więc nie uruchamiam funkcji ponownie.
Wynik jest już w cache, więc nie uruchamiam funkcji ponownie.
Wynik jest już w cache, więc nie uruchamiam funkcji ponownie.
Wynik jest już w cache, więc nie uruchamiam funkcji ponownie.
Wynik jest już w cache, więc nie uruchamiam funkcji ponownie.
Wynik jest już w cache, więc nie uruchamiam funkcji ponownie.
Wynik jest już w cache, więc nie uruchamiam funkcji ponownie.
Wynik jest już w cache, więc nie uruchamiam funkcji ponownie.
Wynik jest już w cache, więc nie uruchamiam funkcji ponownie.
Wynik jest już w cache, więc nie uruchamiam funkcji ponownie.
Wynik jest już w cache, więc nie uruchamiam funkcji ponownie.
Wynik jest już w cache, więc nie uruchamiam funkcji ponownie.
Wynik jest już w cache, więc nie uruchamiam funkcji ponownie.
Wynik jest już w cache, więc nie uruchamiam funkcji ponownie.
Wynik jest już w cache, więc nie uruchamiam funkcji ponownie.
Wynik jest już w cache, więc nie uruchamiam funkcji ponownie.
Wynik je

In [10]:
print("\nPodsumowanie eksperymentu:")
print(f"Udało się: {sukcesy}/100")
print(f"Nie udało się: {porazki}/100")
print(f"Prawdopodobieństwo sukcesu z eksperymentu: {p_empiryczne:.3f}")
print(f"Prawdopodobieństwo teoretyczne: {p_teoretyczne:.3f}")

print("\nWniosek:")
print(f"Przy 5 próbach szansa sukcesu wynosi 1 - 0.5^5, czyli około {p_teoretyczne * 100:.1f}%.")


Podsumowanie eksperymentu:
Udało się: 100/100
Nie udało się: 0/100
Prawdopodobieństwo sukcesu z eksperymentu: 1.000
Prawdopodobieństwo teoretyczne: 0.969

Wniosek:
Przy 5 próbach szansa sukcesu wynosi 1 - 0.5^5, czyli około 96.9%.


### Wnioski

Dekorator `cache_to_disk` zadziałał poprawnie, ponieważ drugie wywołanie tej samej funkcji z tym samym argumentem nie uruchomiło już funkcji ponownie, tylko odczytało wynik z pliku cache. Widać to po komunikacie, że wynik był już zapisany i funkcja nie musiała ponownie symulować pobierania danych.

Dekorator `retry` również spełnił swoje zadanie. Dla niektórych wywołań funkcja zgłaszała błąd, ale dzięki ponawianiu próby udało się ostatecznie uzyskać wynik. Przykładowo dla `id=59` funkcja kilka razy zwróciła błąd, ale kolejne próby pozwoliły uniknąć całkowitego niepowodzenia.

Teoretyczne prawdopodobieństwo sukcesu przy 5 próbach wynosi `1 - 0.5^5`, czyli około 96.9%. W eksperymencie udało się uzyskać 100 sukcesów na 100 wywołań, czyli wynik empiryczny wyniósł 100%. Różnica względem wartości teoretycznej jest naturalna, ponieważ eksperyment był losowy, a dodatkowo część wyników była już dostępna w cache, więc funkcja nie musiała za każdym razem wykonywać niestabilnego kodu.